In [ ]:
import os
from google.colab import userdata
os.environ["ILLINOIS_CHAT_KEY"] = userdata.get("ILLINOIS_CHAT_KEY")

In [ ]:
import requests

url = "https://chat.illinois.edu/api/chat-api/chat"
headers = {"Content-Type": "application/json"}
data = {
    "model": "gpt-oss:120b",
    "messages": [
        {"role": "user", "content": "Reply with exactly: ok"}
    ],
    "api_key": os.environ["ILLINOIS_CHAT_KEY"],
    "course_name": "521-Final-Project",
    "stream": False,
    "temperature": 0.1,
    "retrieval_only": False,
}

r = requests.post(url, headers=headers, json=data, timeout=300)
print(r.status_code)
print(r.text[:2000])

200
{"message":"ok","contexts":[]}


In [ ]:
import os, time, requests

URL = "https://chat.illinois.edu/api/chat-api/chat"
MODEL = "gpt-oss:120b"
COURSE = "521-Final-Project"

def chat(messages, temperature=0.1, retries=3):
    payload = {
        "model": MODEL, "messages": messages,
        "api_key": os.environ["ILLINOIS_CHAT_KEY"],
        "course_name": COURSE, "stream": False,
        "temperature": temperature, "retrieval_only": False,
    }
    for attempt in range(retries):
        try:
            r = requests.post(URL, json=payload, timeout=300)
            r.raise_for_status()
            return r.json()["message"]
        except (requests.RequestException, KeyError, ValueError):
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

In [ ]:
import ast
from dataclasses import dataclass, field, asdict
from typing import Literal, Optional
import torch
import importlib.util, os, tempfile, uuid

Gate = Literal["compile", "test", "volta", "bench"]

@dataclass
class Feedback:
    passed: bool
    gate: Optional[Gate] = None
    diagnostics: list[str] = field(default_factory=list)
    latency_ms: Optional[float] = None

    def to_dict(self):
        return asdict(self)

# def evaluate(kernel_src, shape, use_volta=True):
#     try:
#         ast.parse(kernel_src)
#     except SyntaxError as e:
#         return Feedback(False, "compile", [f"line {e.lineno}: {e.msg}"])
#     if "@triton.jit" not in kernel_src:
#         return Feedback(False, "compile", ["no @triton.jit kernel found"])
#     return Feedback(True, latency_ms=1.0)

SHAPES = [(128, 1024), (4, 7), (1, 1), (16, 3000)]

def load_module(kernel_src):
    name = f"cand_{uuid.uuid4().hex}"
    path = os.path.join(tempfile.mkdtemp(), name + ".py")
    with open(path, "w") as f:
        f.write(kernel_src)
    spec = importlib.util.spec_from_file_location(name, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

def evaluate(kernel_src, shape=None, use_volta=True):
    try:
        mod = load_module(kernel_src)
    except Exception as e:
        return Feedback(False, "compile", [f"{type(e).__name__}: {e}"[:1500]])
    fn = getattr(mod, "softmax", None)
    if fn is None or type(fn).__name__ == "JITFunction":
        return Feedback(False, "compile", [
            "module must define a plain Python function `softmax(x)` that "
            "takes a CUDA tensor, launches the kernel, and returns the "
            "output tensor; the @triton.jit kernel needs a different name"])
    for M, N in SHAPES:
        x = torch.randn(M, N, device="cuda")
        ref = torch.softmax(x, dim=1)
        try:
            y = fn(x)
        except Exception as e:
            return Feedback(False, "compile",
                            [f"shape {(M, N)}: {type(e).__name__}: {e}"[:1500]])
        if y.shape != ref.shape:
            return Feedback(False, "test",
                            [f"shape {(M, N)}: output shape {tuple(y.shape)}"])
        if not torch.allclose(y, ref, atol=1e-5):
            err = (y - ref).abs().max().item()
            return Feedback(False, "test",
                            [f"shape {(M, N)}: max abs error {err:.3e}"])
    return Feedback(True)

In [ ]:
import re, json

SYSTEM = ("You write Triton GPU kernels. Reply with exactly one ```python "
          "code block containing the whole module: a @triton.jit kernel, "
          "and a plain Python function `softmax(x)` that launches it and "
          "returns the output tensor. No text outside the block.")

TASK = """Reference (x is a float32 CUDA tensor of shape (M, N)):

def softmax(x):
    return torch.softmax(x, dim=1)

Write an equivalent Triton implementation."""

def extract_code(reply):
    blocks = re.findall(r"```[^\n]*\n(.*?)```", reply, re.DOTALL)
    jit = [b for b in blocks if "@triton.jit" in b]
    return max(jit or blocks, key=len) if blocks else None

def run(llm=chat, max_iters=5, shape=(128, 1024),
        use_volta=True, log_path="run.jsonl"):
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": TASK}]
    for i in range(max_iters):
        reply = llm(messages)
        src = extract_code(reply)
        if src is None:
            fb = Feedback(False, "compile", ["no code block in reply"])
        else:
            fb = evaluate(src, shape, use_volta)
        with open(log_path, "a") as f:
            f.write(json.dumps({"iter": i, "reply": reply, "src": src,
                    "feedback": fb.to_dict()}) + "\n")
        if fb.passed:
            return src, i + 1
        messages.append({"role": "assistant", "content": reply})
        messages.append({"role": "user", "content":
            f"Gate `{fb.gate}` failed:\n" + "\n".join(fb.diagnostics) +
            "\nFix the kernel and reply with the full code again."})
    return None, max_iters

In [ ]:
# replies = iter([
#     "here you go",                                    # no code block
#     "```python\ndef softmax(x):\n    return x\n```",  # no @triton.jit
#     "```python\n@triton.jit\ndef k(): pass\n```",     # passes stub
# ])
# src, n = run(llm=lambda msgs: next(replies), log_path="test.jsonl")
# assert n == 3 and "@triton.jit" in src
# print(open("test.jsonl").read())

In [ ]:
# src, n = run(log_path="real.jsonl")
# print("iterations:", n)
# print(src)

In [ ]:
src, n = run(log_path="real2.jsonl")
print("iterations:", n)
print(open("real2.jsonl").read())

iterations: 2
{"iter": 0, "src": "import triton\nimport triton.language as tl\nimport torch\n\n\n@triton.jit\ndef softmax_kernel(\n    X_ptr,               # input   (M\u00d7N) matrix\n    Y_ptr,               # output  (M\u00d7N) matrix\n    M, N,\n    BLOCK_M: tl.constexpr,   # rows per program\u2011block\n    BLOCK_N: tl.constexpr,   # columns per program\u2011block\n):\n    \"\"\"\n    Row\u2011wise softmax over a 2\u2011D tensor of shape (M, N).\n    \"\"\"\n\n    # ------------------------------------------------------------------\n    # 1\ufe0f\u20e3 Compute the row index this program block works on.\n    # ------------------------------------------------------------------\n    pid = tl.program_id(0)                     # linear program id\n    offs_m = pid * BLOCK_M + tl.arange(0, BLOCK_M)\n    mask_m = offs_m < M                         # guard against OOB rows\n\n    # ------------------------------------------------------------------\n    # 2\ufe0f\u20e3 Load a tile of the i

In [ ]:
%%writefile baseline.py
import torch, triton
import triton.language as tl

@triton.jit
def softmax_kernel(x_ptr, y_ptr, stride, n_cols, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK)
    mask = cols < n_cols
    x = tl.load(x_ptr + row * stride + cols, mask=mask, other=-float("inf"))
    e = tl.exp(x - tl.max(x, axis=0))
    y = e / tl.sum(e, axis=0)
    tl.store(y_ptr + row * stride + cols, y, mask=mask)

def softmax(x):
    x = x.contiguous()
    M, N = x.shape
    y = torch.empty_like(x)
    softmax_kernel[(M,)](x, y, x.stride(0), N,
                         BLOCK=triton.next_power_of_2(N))
    return y

Overwriting baseline.py


In [ ]:
print(evaluate(open("baseline.py").read()))

Feedback(passed=True, gate=None, diagnostics=[], latency_ms=None)


In [ ]:
bad = open("baseline.py").read().replace("triton.next_power_of_2(N)", "1024")
print(evaluate(bad))

Feedback(passed=False, gate='test', diagnostics=['shape (16, 3000): max abs error 4.354e+00'], latency_ms=None)


In [ ]:
import time

open("agent_softmax.py", "w").write(src)
base, agent = load_module(open("baseline.py").read()), load_module(src)

def bench(fn, shape=(4096, 1024), reps=50):
    x = torch.randn(*shape, device="cuda")
    for _ in range(5):
        fn(x)                      # warmup; first call includes JIT compile
    torch.cuda.synchronize()
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(x)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - t0) * 1e3)
    return sorted(times)[reps // 2]

for name, fn in [("torch", lambda x: torch.softmax(x, dim=1)),
                 ("baseline", base.softmax), ("agent", agent.softmax)]:
    print(f"{name:9s}{bench(fn):.3f} ms")

torch    0.170 ms
baseline 0.180 ms
agent    0.397 ms


In [ ]:
x = torch.randn(128, 1024, device="cuda")
y = torch.empty_like(x)
k = base.softmax_kernel[(128,)](x, y, x.stride(0), 1024, BLOCK=1024)
ptx = k.asm["ptx"]
open("baseline.ptx", "w").write(ptx)
print(len(ptx.splitlines()), "lines")
print(ptx[:600])

506 lines
//
// Generated by LLVM NVPTX Back-End
//

.version 8.8
.target sm_75
.address_size 64

	// .globl	softmax_kernel          // -- Begin function softmax_kernel
.extern .shared .align 16 .b8 global_smem[];
                                        // @softmax_kernel
.visible .entry softmax_kernel(
	.param .u64 .ptr .global .align 1 softmax_kernel_param_0,
	.param .u64 .ptr .global .align 1 softmax_kernel_param_1,
	.param .u32 softmax_kernel_param_2,
	.param .u32 softmax_kernel_param_3,
	.param .u64 .ptr .global .align 1 softmax_kernel_param_4,
	.param .u64 .ptr .global .align 1 softmax_kernel_para


In [ ]:
k2 = agent._softmax_kernel[(128,)](x, y, 128, 1024, BLOCK_N=128)
open("agent.ptx", "w").write(k2.asm["ptx"])
print(len(k2.asm["ptx"].splitlines()), "lines")

507 lines


In [ ]:
!git clone https://github.com/willtunnels/volta
!ls volta
!head -120 volta/README.md

Cloning into 'volta'...
remote: Enumerating objects: 806, done.
remote: Counting objects: 100% (215/215), done.
remote: Compressing objects: 100% (68/68), done.
remote: Total 806 (delta 167), reused 158 (delta 147), pack-reused 591 (from 1)
Receiving objects: 100% (806/806), 1.30 MiB | 12.44 MiB/s, done.
Resolving deltas: 100% (430/430), done.
Cargo.lock  CLAUDE.md  docs	README.md  scripts
Cargo.toml  crates     LICENSE	results
# Volta

Volta is a data race and equivalence checker for NVIDIA GPU kernels, implementing "Equivalence Checking of ML GPU Kernels" (OOPSLA 2026). Given a reference kernel and an optimized counterpart, Volta proves them semantically equivalent over the reals — identical outputs on all valid inputs, with floating-point values modeled as real numbers — thereby verifying the correctness of the optimized kernel.

## Relationship to the paper

The evaluation in the published version of the paper was produced with this implementation. The raw results live in [`results

In [ ]:
xs = torch.randn(1, 64, device="cuda")
ys = torch.empty_like(xs)
kb = base.softmax_kernel[(1,)](xs, ys, xs.stride(0), 64, BLOCK=64)
ka = agent._softmax_kernel[(1,)](xs, ys, 1, 64, BLOCK_N=16)
for name, k in [("baseline", kb), ("agent", ka)]:
    open(f"{name}_small.ptx", "w").write(k.asm["ptx"])
    print(name, k.metadata.num_warps, k.metadata.shared)
!grep -n -A 12 "visible .entry" baseline_small.ptx agent_small.ptx

baseline 4 8
agent 4 0
baseline_small.ptx:12:.visible .entry softmax_kernel(
baseline_small.ptx-13-	.param .u64 .ptr .global .align 1 softmax_kernel_param_0,
baseline_small.ptx-14-	.param .u64 .ptr .global .align 1 softmax_kernel_param_1,
baseline_small.ptx-15-	.param .u32 softmax_kernel_param_2,
baseline_small.ptx-16-	.param .u32 softmax_kernel_param_3,
baseline_small.ptx-17-	.param .u64 .ptr .global .align 1 softmax_kernel_param_4,
baseline_small.ptx-18-	.param .u64 .ptr .global .align 1 softmax_kernel_param_5
baseline_small.ptx-19-)
baseline_small.ptx-20-.reqntid 128
baseline_small.ptx-21-{
baseline_small.ptx-22-	.reg .pred 	%p<4>;
baseline_small.ptx-23-	.reg .b32 	%r<47>;
baseline_small.ptx-24-	.reg .b64 	%rd<9>;
--
agent_small.ptx:11:.visible .entry _softmax_kernel(
agent_small.ptx-12-	.param .u64 .ptr .global .align 1 _softmax_kernel_param_0,
agent_small.ptx-13-	.param .u64 .ptr .global .align 1 _softmax_kernel_param_1,
agent_small.ptx-14-	.param .u32 _softmax_kernel_param_2,
age

In [ ]:
!curl -sSf https://sh.rustup.rs | sh -s -- -y -q
!cd volta && ~/.cargo/bin/cargo build --release 2>&1 | tail -5


  stable-x86_64-unknown-linux-gnu installed - rustc 1.100.0 (0ef1bc386 2026-11-09)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source the
corresponding env file under $HOME/.cargo.

Consider running the right command for your shell (note the leading DOT):
. "$HOME/.cargo/env" # For sh/ash/dash/pdksh/bash
cargo:rerun-if-env-changed=CC_x86_64-unknown-linux-gnu
CC_x86_64-unknown-linux-gnu = None
cargo:rerun-if-env-changed=CC_x86_64_unknown_linux_gnu
CC_x86_64_unknown_linux_gnu = None
cargo:rerun-if-env-changed=HOST_CC
HOST_CC = None
cargo:rerun-if-env-changed=CC
CC = None
cargo:rerun-if-env-changed=CC_ENABLE_DEBUG_OUTPUT
cargo:rerun-if-env-changed=CPATH
cargo:rerun-if-env-changed=C_INCLUDE_PATH
cargo:rerun-if-env-changed=CPLUS_INCLUDE_PATH
cargo:rerun-if-env-changed=OBJC_INCLUDE_PATH
cargo:rerun

In [ ]:
!apt-get install -y -qq libz3-dev
!cd volta && ~/.cargo/bin/cargo build --release 2>&1 | tail -3
!ls -la volta/target/release/volta

Selecting previously unselected package libz3-4:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libz3-4_4.8.12-3.1build1_amd64.deb ...
Unpacking libz3-4:amd64 (4.8.12-3.1build1) ...
Selecting previously unselected package libz3-dev:amd64.
Preparing to unpack .../libz3-dev_4.8.12-3.1build1_amd64.deb ...
Unpacking libz3-dev:amd64 (4.8.12-3.1build1) ...
Setting up libz3-4:amd64 (4.8.12-3.1build1) ...
Setting up libz3-dev:amd64 (4.8.12-3.1build1) ...
Processing triggers for libc-bin (2.39-0ubuntu8.9) ...
/sbin/ldconfig.real: /usr/local/lib/libtcm.so.1 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libtbbbind_2_0.so.3 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_loader.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_opencl.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/local/lib/libur_adapter_level_zero_v2.so.0 is not a symbolic link

/sbin/ldconfig.real: /usr/loc

In [ ]:
!volta/target/release/volta parse baseline_small.ptx | tail -5
!volta/target/release/volta parse agent_small.ptx | tail -5

log: volta-logs/1795049224-50077-parse.log
Parsed successfully: baseline_small.ptx
  Entries (kernels): 1
log: volta-logs/1795049224-50084-parse.log
Parsed successfully: agent_small.ptx
  Entries (kernels): 1


In [ ]:
%%bash
volta/target/release/volta analyze baseline_small.ptx -k softmax_kernel \
  -b 128 -g 1 \
  --array "x:0x100000000:4:64:in" \
  --array "y:0x200000000:4:64:out" \
  --array "scratch:0x300000000:1:64:in" \
  --param ptr:x --param ptr:y --param int:64 --param int:64 \
  --param ptr:scratch --param ptr:scratch \
  --dyn-shared 8 --no-profile 2>&1 | tail -30

analysis failed: lowering failed: Invalid operand: Vector([Ident(AsciiString { data: ['%', 'r', '1'] })]) in ld
Reason: the .vN width modifier must match the vector operand's register count
log: volta-logs/1795049226-50091-analyze.log


In [ ]:
!grep -n "ld\." baseline_small.ptx | head -40

30:	ld.param.b64 	%rd3, [softmax_kernel_param_0];
31:	ld.param.b64 	%rd4, [softmax_kernel_param_1];
35:	ld.param.b32 	%r8, [softmax_kernel_param_2];
39:	ld.param.b32 	%r11, [softmax_kernel_param_3];
54:	@%p1 ld.global.b32 { %r1 }, [ %rd1 + 0 ];
103:	ld.shared.b32 	%r28, [%r27];
154:	ld.shared.b32 	%r43, [%r27];


In [ ]:
%%bash
volta/target/release/volta analyze agent_small.ptx -k _softmax_kernel \
  -b 128 -g 1 \
  --array "x:0x100000000:4:64:in" \
  --array "y:0x200000000:4:64:out" \
  --array "scratch:0x300000000:1:64:in" \
  --param ptr:x --param ptr:y --param int:64 \
  --param ptr:scratch --param ptr:scratch \
  --no-profile 2>&1 | tail -30

analysis failed: lowering failed: Invalid operand: Vector([Ident(AsciiString { data: ['%', 'r', '7'] })]) in ld
Reason: the .vN width modifier must match the vector operand's register count
log: volta-logs/1795049284-50353-analyze.log


In [ ]:
%%bash
for k in baseline agent; do
  sed -E 's/\{ *(%[a-z]+[0-9]+) *\}/\1/g' ${k}_small.ptx > ${k}_fix.ptx
done
grep -n "ld.global\|st.global" baseline_fix.ptx
volta/target/release/volta analyze baseline_fix.ptx -k softmax_kernel \
  -b 128 -g 1 \
  --array "x:0x100000000:4:64:in" \
  --array "y:0x200000000:4:64:out" \
  --array "scratch:0x300000000:1:64:in" \
  --param ptr:x --param ptr:y --param int:64 --param int:64 \
  --param ptr:scratch --param ptr:scratch \
  --dyn-shared 8 --no-profile 2>&1 | tail -30

54:	@%p1 ld.global.b32 %r1, [ %rd1 + 0 ];
170:	@%p2 st.global.b32 [ %rd2 + 0 ], %r6;
analysis failed: lowering failed: Unsupported instruction: Ex2(Float32 { ftz: false, dst: Ident(AsciiString { data: ['%', 'r', '3', '3'] }), src: Ident(AsciiString { data: ['%', 'r', '3', '2'] }) })
Reason: Instruction not yet implemented
log: volta-logs/1795049346-50634-analyze.log
